# Creating a agent that takes track of product availability in certain warehouses

In [1]:
import random
from qdrant_client import QdrantClient
import psycopg2
from psycopg2.extras import RealDictCursor, execute_batch
import numpy as np
from qdrant_client import QdrantClient

In [2]:
# filling the newly created warehouse table with some additional data on warehouses
warehouses = [
    {
        "warehouse_id": "DE-BER-01",
        "warehouse_location": "Berlin, Germany",
        "warehouse_name": "Berlin Distribution Center"
    },
    {
        "warehouse_id": "DE-MUN-01",
        "warehouse_location": "Munich, Germany",
        "warehouse_name": "Munich Logistics Hub"
    },
    {
        "warehouse_id": "DE-HAM-01",
        "warehouse_location": "Hamburg, Germany",
        "warehouse_name": "Hamburg North Warehouse"
    },
    {
        "warehouse_id": "FR-PAR-01",
        "warehouse_location": "Paris, France",
        "warehouse_name": "Paris Central Depot"
    },
    {
        "warehouse_id": "FR-LYO-01",
        "warehouse_location": "Lyon, France",
        "warehouse_name": "Lyon Regional Warehouse"
    },
    {
        "warehouse_id": "FR-MAR-01",
        "warehouse_location": "Marseille, France",
        "warehouse_name": "Marseille Mediterranean Hub"
    }
]

### Simulate Stock Availability for each of the warehouses
Retrieve all item IDs from the Amamzon items Qdrant Collection and give them some random stock

In [3]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [ ]:
# dummy query vector to get all the data
dummy_vector = np.zeros(1536).tolist()

In [6]:
payload = qdrant_client.query_points(
    collection_name="Amazon-items-collection-hybrid-search",
    query=dummy_vector,
    using="text-embedding-3-small",
    limit=50, #all
    with_payload=["parent_asin"],
    with_vectors=False
).points

In [7]:
payload

[ScoredPoint(id=9, version=3, score=0.0, payload={'parent_asin': 'B0B76GG8L1'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=21, version=3, score=0.0, payload={'parent_asin': 'B09VDLH5M6'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=45, version=3, score=0.0, payload={'parent_asin': 'B0BC4PGXFK'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=28, version=3, score=0.0, payload={'parent_asin': 'B09PTX6461'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=16, version=3, score=0.0, payload={'parent_asin': 'B0C8S6BBY9'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=39, version=3, score=0.0, payload={'parent_asin': 'B0B6R7CKVP'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=35, version=3, score=0.0, payload={'parent_asin': 'B09TFM1SFQ'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=33, version=3, score=0.0, payload={'parent_asin': 'B09QGRRY7G'}, vector=None, shar

In [8]:
len(payload)

50

In [9]:
parent_asin_list = [item.payload["parent_asin"] for item in payload]

In [10]:
parent_asin_list

['B0B76GG8L1',
 'B09VDLH5M6',
 'B0BC4PGXFK',
 'B09PTX6461',
 'B0C8S6BBY9',
 'B0B6R7CKVP',
 'B09TFM1SFQ',
 'B09QGRRY7G',
 'B09XSBZQVS',
 'B0B5CYWHWV',
 'B0CFHWF326',
 'B0BMFYWK6T',
 'B0BZ944K9M',
 'B09KQP2H7N',
 'B09WCT9S1R',
 'B0BZ44Y4C6',
 'B0BBF2VC6X',
 'B0BXC72RLD',
 'B0BZJP91PF',
 'B0C996WY16',
 'B0BG6TMXDD',
 'B0BM9THPDQ',
 'B0828S2KMV',
 'B0BB6TFQ22',
 'B0BYRWPV86',
 'B09P4QW5Y2',
 'B0CF57H28T',
 'B0C9QZS95R',
 'B0BGH3H1WM',
 'B0BGPPX7DC',
 'B09YHG2Z7F',
 'B09X9838WY',
 'B0BN1CMWCP',
 'B0C9ZWCZ99',
 'B0B8SC1T45',
 'B099N9F3FP',
 'B0BM657X74',
 'B09R1S39N5',
 'B0BBVJJRHD',
 'B0C8DBH7ZT',
 'B0BRV544MV',
 'B09RMJWBSM',
 'B0CC4HBS85',
 'B0BYD7PGV1',
 'B0CH8DRD6K',
 'B0B8ZMQ53J',
 'B09V2Y1TQB',
 'B0B5LW6277',
 'B0BRJS644Z',
 'B0C4NJPN4Q']

### Generate synthetic availability of the stock

In [11]:
def generate_inventory_data(warehouses, product_ids, availability_rate=0.75):
    
    inventory_records = []
    
    for warehouse in warehouses:
        for product_id in product_ids:
            # 75% chance the product is available in this warehouse
            if random.random() < availability_rate:
                total_quantity = random.randint(0, 100)
                
                # Only add to inventory if quantity > 0
                if total_quantity > 0:
                    inventory_records.append({
                        "warehouse_id": warehouse["warehouse_id"],
                        "warehouse_location": warehouse["warehouse_location"],
                        "warehouse_name": warehouse["warehouse_name"],
                        "product_id": product_id,
                        "total_quantity": total_quantity,
                        "reserved_quantity": 0  # Starting with no reservations
                    })
    
    return inventory_records

In [12]:
inventory_data = generate_inventory_data(warehouses, parent_asin_list)

In [13]:
inventory_data

[{'warehouse_id': 'DE-BER-01',
  'warehouse_location': 'Berlin, Germany',
  'warehouse_name': 'Berlin Distribution Center',
  'product_id': 'B0B76GG8L1',
  'total_quantity': 28,
  'reserved_quantity': 0},
 {'warehouse_id': 'DE-BER-01',
  'warehouse_location': 'Berlin, Germany',
  'warehouse_name': 'Berlin Distribution Center',
  'product_id': 'B09VDLH5M6',
  'total_quantity': 42,
  'reserved_quantity': 0},
 {'warehouse_id': 'DE-BER-01',
  'warehouse_location': 'Berlin, Germany',
  'warehouse_name': 'Berlin Distribution Center',
  'product_id': 'B0BC4PGXFK',
  'total_quantity': 22,
  'reserved_quantity': 0},
 {'warehouse_id': 'DE-BER-01',
  'warehouse_location': 'Berlin, Germany',
  'warehouse_name': 'Berlin Distribution Center',
  'product_id': 'B09PTX6461',
  'total_quantity': 68,
  'reserved_quantity': 0},
 {'warehouse_id': 'DE-BER-01',
  'warehouse_location': 'Berlin, Germany',
  'warehouse_name': 'Berlin Distribution Center',
  'product_id': 'B0C8S6BBY9',
  'total_quantity': 56,
  

In [14]:
len(inventory_data)

227

### Write synthetic data to DB

In [17]:
def insert_inventory_to_db(inventory_records):
   
    try:
        # Connect to the database
        conn = psycopg2.connect(
            host="localhost",
            port=5432,
            database="tools_database",
            user="tool_user",
            password="tool_user_password"
        )
        conn.autocommit = True

        with conn.cursor(cursor_factory=RealDictCursor) as cursor:
        
            # Prepare the INSERT query
            insert_query = """
            INSERT INTO warehouses.inventory 
            (warehouse_id, warehouse_location, warehouse_name, product_id, total_quantity, reserved_quantity)
            VALUES (%(warehouse_id)s, %(warehouse_location)s, %(warehouse_name)s, %(product_id)s, %(total_quantity)s, %(reserved_quantity)s)
            """
            
            # Use execute_batch for better performance with many inserts
            execute_batch(cursor, insert_query, inventory_records, page_size=100)
            
            # Commit the transaction
            conn.commit()
            
            print(f"Successfully inserted {len(inventory_records)} records into warehouses.inventory")
            
            # Close cursor and connection
            cursor.close()
            conn.close()
        
    except psycopg2.Error as e:
        print(f"Database error: {e}")
        if conn:
            conn.rollback()
    except Exception as e:
        print(f"Error: {e}")
    finally:
        if cursor:
            cursor.close()
        if conn:
            conn.close()

In [18]:
insert_inventory_to_db(inventory_data)

Successfully inserted 227 records into warehouses.inventory
